## 🎯 **Obiettivo di questo esercizio**

Imparare a **"conoscere"** una tabella prima di toccarla:
- Quante righe? Quante colonne?
- Che tipi di dati? Quanti NULL?
- Quali pattern di sporcizia esistono?
- Quali valori anomali (outlier)?
---
## 📋 **Metodi e funzioni (riassunto)**

### **In SQLite (esplorazione)**

| Funzione | Cosa fa | Esempio |
|----------|---------|---------|
| `SELECT COUNT(*)` | Numero righe | `SELECT COUNT(*) FROM tabella;` |
| `PRAGMA table_info(tabella)` | Struttura colonne | `PRAGMA table_info(sondaggi);` |
| `SELECT DISTINCT colonna` | Valori unici | `SELECT DISTINCT Sito FROM sondaggi;` |
| `SELECT COUNT(DISTINCT colonna)` | Conteggio valori unici | `SELECT COUNT(DISTINCT Sito) FROM sondaggi;` |
| `colonna IS NULL` | Trovare NULL | `SELECT COUNT(*) FROM tabella WHERE colonna IS NULL;` |
| `MIN()`, `MAX()`, `AVG()` | Statistiche base | `SELECT MIN(profondità), MAX(profondità), AVG(profondità) FROM sondaggi;` |
| `colonna GLOB '[0-9]*'` | Pattern matching | Trova valori che iniziano con numeri |
| `colonna LIKE '%[a-z]%'` | Trova lettere minuscole | `WHERE Sito LIKE '%[a-z]%'` |

## 🎯 **Quesiti (esplorazione SOLO, nessuna pulizia)**

Usando **SQL** o **pandas** (scegli tu), rispondi a queste domande:

### **1. Panoramica generale**
- Quante righe ha la tabella?
- Quante colonne? Quali sono i loro nomi?
- Quali colonne sono numeriche? Quali sono testo?

### **2. Analisi NULL**
- Quanti NULL ci sono in `Au_ppm`?
- Quanti NULL in `Profondita_m`?
- Quale percentuale del totale rappresentano?

### **3. Pattern di sporcizia su ID_Campione**
- Quanti ID sono in MAIUSCOLO?
- Quanti in minuscolo?
- Quanti hanno spazi?
- Quanti hanno il trattino `-`?
- Quanti NON hanno il trattino?

### **4. Pattern di sporcizia su Sito**
- Elenca tutti i valori unici (compresi quelli sporchi)
- Quanti sono in MAIUSCOLO? Minuscolo?
- Quanti hanno spazi?

### **5. Analisi Profondita_m**
- Quali valori non sono numerici? (es. "zero", "ND", "mille")
- Quali sono outlier evidenti? (valori negativi o > 1000)
- Qual è la media, mediana, min, max dei valori puliti (escludendo outlier e non-numerici)?

### **6. Analisi Au_ppm**
- Quali valori non sono numerici? ("ND", "traccia")
- Quali sono outlier? (valori > 10 ppm)
- Distribuzione: media, mediana, min, max (escludendo outlier e non-numerici)

### **7. Analisi Data**
- Quanti formati di data diversi vedi?
- Elenca i valori unici della colonna Data
- Quali record hanno "oggi"?

---

## 📝 **Consegna**

Crea un notebook con:
1. Il codice per caricare il dataset
2. Le tue query/comandi di esplorazione
3. Per ogni domanda, scrivi:
   - Il codice usato
   - Il risultato ottenuto
   - Una breve nota su cosa quel risultato ti dice

**Non scrivere ancora codice di pulizia!** Solo esplorazione.

In [6]:
import sqlite3
import pandas as pd
import random
import numpy as np

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Genera 50 campioni con vari pattern di sporcizia
np.random.seed(42)
id_campione = [f"CAMP-{i:03d}" for i in range(1, 51)]
# Aggiungi sporcizia a caso
for i in range(len(id_campione)):
    if i % 7 == 0:
        id_campione[i] = id_campione[i].lower()
    elif i % 11 == 0:
        id_campione[i] = id_campione[i].replace("-", "")
    elif i % 13 == 0:
        id_campione[i] = " " + id_campione[i] + " "

siti = ["Nord", "Sud", "Est", "Ovest", "Centrale"]
sito = [random.choice(siti) for _ in range(50)]
# Aggiungi sporcizia
for i in range(len(sito)):
    if i % 5 == 0:
        sito[i] = sito[i].lower()
    elif i % 17 == 0:
        sito[i] = sito[i] + " "

profondita = [round(random.uniform(10, 300), 1) for _ in range(50)]
# Aggiungi valori anomali e sporcizia
profondita[3] = "zero"
profondita[12] = "ND"
profondita[25] = -50  # outlier
profondita[38] = "mille"
profondita[42] = 9999  # outlier

au_ppm = [round(random.uniform(0, 5), 2) for _ in range(50)]
au_ppm[7] = "ND"
au_ppm[14] = "traccia"
au_ppm[28] = 99.9  # outlier estremo
au_ppm[35] = None

data = [f"2024-{random.randint(1,12):02d}-{random.randint(1,28):02d}" for _ in range(50)]
# Aggiungi formati data sporchi
data[5] = "15/06/2024"
data[19] = "2024.09.20"
data[33] = "20-10-2024"
data[47] = "oggi"

# Crea DataFrame
df_sporco = pd.DataFrame({
    'ID_Campione': id_campione,
    'Sito': sito,
    'Profondita_m': profondita,
    'Au_ppm': au_ppm,
    'Data': data
})

df_sporco.to_sql('campioni', conn, index=False, if_exists='replace')

print("✅ Dataset creato! 50 righe con sporcizia realistica.")
print("\nPrime 10 righe:")
print(df_sporco.head(10))
print(f"\nShape: {df_sporco.shape}")

✅ Dataset creato! 50 righe con sporcizia realistica.

Prime 10 righe:
  ID_Campione      Sito Profondita_m Au_ppm        Data
0    camp-001       sud         39.7   4.88  2024-01-25
1    CAMP-002       Est        130.3   2.57  2024-02-25
2    CAMP-003       Est        184.0   1.11  2024-12-15
3    CAMP-004      Nord         zero   4.47  2024-04-23
4    CAMP-005  Centrale        274.8   1.55  2024-02-02
5    CAMP-006       est        291.9    2.0  15/06/2024
6    CAMP-007       Est         35.3   3.31  2024-06-03
7    camp-008  Centrale        245.4     ND  2024-10-08
8    CAMP-009  Centrale        128.0   4.82  2024-01-07
9    CAMP-010     Ovest        112.5   1.49  2024-09-21

Shape: (50, 5)


In [7]:
print("==================================")
print("--------------Punto 1-------------")
print("==================================")
print("\n")
# -------------------------------------

q_count = """
SELECT 
COUNT(*) AS totale_righe 
FROM campioni;
"""

df_count = pd.read_sql_query(q_count, conn)
print("-------Numero righe-------")
print(df_count)
print("\n")
# -------------------------------------

q_schema = """
PRAGMA table_info(campioni);
"""

print("==================================")
print("--------------Punto 2-------------")
print("==================================")
print("\n")
# -------------------------------------

df_schema = pd.read_sql_query(q_schema, conn)
print("-------Dettagli Tabella-------")
print(df_schema)
print("\n")
# -------------------------------------

q_NULL = """
WITH stats AS (
    SELECT
        COUNT(*) AS totale_righe,
        SUM(CASE WHEN Au_ppm IS NULL THEN 1 ELSE 0 END) AS totale_null_au_ppm,
        SUM(CASE WHEN Profondita_m IS NULL THEN 1 ELSE 0 END) AS totale_null_profondita_m
    FROM campioni
)
SELECT
    totale_null_au_ppm,
    ROUND(totale_null_au_ppm * 100.0 / totale_righe, 2) AS pct_null_au_ppm,
    totale_null_profondita_m,
    ROUND(totale_null_profondita_m * 100.0 / totale_righe, 2) AS pct_null_profondita_m
FROM stats;
"""

df_NULL = pd.read_sql_query(q_NULL, conn)
print("-------Quanti NULL ci sono in Au_ppm / Profondita_m?-------")
print(df_NULL)
print("\n")
# -------------------------------------

print("==================================")
print("--------------Punto 3-------------")
print("==================================")
print("\n")
# -------------------------------------

q_punto_3 = """
SELECT
    SUM(
        CASE
            WHEN ID_Campione IS NOT NULL
             AND ID_Campione = UPPER(ID_Campione)
             AND ID_Campione GLOB '*[A-Z]*'
            THEN 1 ELSE 0
        END
    ) AS totale_ID_maiuscolo,

    SUM(
        CASE
            WHEN ID_Campione IS NOT NULL
             AND ID_Campione = LOWER(ID_Campione)
             AND ID_Campione GLOB '*[a-z]*'
            THEN 1 ELSE 0
        END
    ) AS totale_ID_minuscolo,

    SUM(
        CASE
            WHEN ID_Campione IS NOT NULL
             AND ID_Campione LIKE '% %'
            THEN 1 ELSE 0
        END
    ) AS totale_spazi,

    SUM(
        CASE
            WHEN ID_Campione IS NOT NULL
             AND ID_Campione LIKE '%-%'
            THEN 1 ELSE 0
        END
    ) AS totale_trattino,

    SUM(
        CASE
            WHEN ID_Campione IS NOT NULL
             AND ID_Campione NOT LIKE '%-%'
            THEN 1 ELSE 0
        END
    ) AS totale_NON_trattino
FROM campioni;
"""

df_punto_3 = pd.read_sql_query(q_punto_3, conn)
print("-------Pattern di sporcizia su ID_Campione-------")
print(df_punto_3)
print("\n")
# -------------------------------------

print("==================================")
print("--------------Punto 4-------------")
print("==================================")
print("\n")
# -------------------------------------

q_punto_4 = """
WITH siti_unici AS (
    SELECT DISTINCT Sito
    FROM campioni
)
SELECT
    COUNT(CASE
        WHEN Sito IS NOT NULL
         AND Sito = UPPER(Sito)
         AND Sito GLOB '*[A-Z]*'
        THEN 1
    END) AS valori_MAIUSCOLO,

    COUNT(CASE
        WHEN Sito IS NOT NULL
         AND Sito = LOWER(Sito)
         AND Sito GLOB '*[a-z]*'
        THEN 1
    END) AS valori_minuscolo,

    COUNT(CASE
        WHEN Sito IS NOT NULL
         AND Sito LIKE '% %'
        THEN 1
    END) AS valori_spazio
FROM siti_unici;
"""

df_punto_4 = pd.read_sql_query(q_punto_4, conn)
print("-------Pattern di sporcizia su Sito-------")
print(df_punto_4)
print("\n")
# -------------------------------------

q_valori_unici = """
SELECT DISTINCT Sito
FROM campioni
ORDER BY Sito;
"""

df_valori_unici = pd.read_sql_query(q_valori_unici, conn)
print("-------Valori unici di Sito-------")
print(df_valori_unici)
print()
# -------------------------------------

print("==================================")
print("--------------Punto 5-------------")
print("==================================")
print("\n")
# -------------------------------------

q_punto_5A = """
SELECT 
Profondita_m, 
COUNT(*) AS n
FROM campioni
GROUP BY Profondita_m
ORDER BY n DESC, Profondita_m;
"""

df_punto_5A = pd.read_sql_query(q_punto_5A, conn)
print("-------Analisi Profondita_m-------")
print(df_punto_5A)
print()
# -------------------------------------

q_punto_5B = """
SELECT 
LOWER(TRIM(Profondita_m)) AS valore, 
COUNT(*) AS n
FROM campioni
WHERE TRIM(Profondita_m) <> ''
  AND TRIM(Profondita_m) GLOB '*[^0-9.]*'
GROUP BY LOWER(TRIM(Profondita_m))
ORDER BY n DESC, valore;
"""

df_punto_5B = pd.read_sql_query(q_punto_5B, conn)
print("-------Valori non numerici-------")
print(df_punto_5B)
print()
# -------------------------------------

q_punto_5C = """
SELECT
    SUM(CASE WHEN Profondita_m IS NULL THEN 1 ELSE 0 END) AS nulli,
    SUM(CASE WHEN TRIM(Profondita_m) = '' THEN 1 ELSE 0 END) AS vuoti
FROM campioni;
"""

df_punto_5C = pd.read_sql_query(q_punto_5C, conn)
print("-------Null Values Profondita_m-------")
print(df_punto_5C)
print()
# -------------------------------------

q_punto_5D = """
SELECT Profondita_m,
       CAST(REPLACE(TRIM(Profondita_m), ',', '.') AS REAL) AS val_num
FROM campioni
WHERE TRIM(Profondita_m) NOT GLOB '*[^0-9.]*'
  AND (
      CAST(REPLACE(TRIM(Profondita_m), ',', '.') AS REAL) < 0
      OR
      CAST(REPLACE(TRIM(Profondita_m), ',', '.') AS REAL) > 1000
  )
ORDER BY val_num;
"""

df_punto_5D = pd.read_sql_query(q_punto_5D, conn)
print("-------Outlier Profondita_m-------")
print(df_punto_5D)
print()
# -------------------------------------

q_punto_5E = """
WITH base AS (
    SELECT
        Profondita_m,
        LOWER(TRIM(Profondita_m)) AS v
    FROM campioni
),
parsed AS (
    SELECT
        CASE
            WHEN v IS NULL OR v = '' THEN NULL
            WHEN v GLOB '*[^0-9,.-]*' THEN NULL
            ELSE CAST(REPLACE(v, ',', '.') AS REAL)
        END AS valore_num
    FROM base
),
clean AS (
    SELECT valore_num
    FROM parsed
    WHERE valore_num BETWEEN 0 AND 1000
),
ordered AS (
    SELECT
        valore_num,
        ROW_NUMBER() OVER (ORDER BY valore_num) AS rn,
        COUNT(*) OVER () AS n
    FROM clean
)
SELECT
    (SELECT MIN(valore_num) FROM clean) AS minimo,
    (SELECT MAX(valore_num) FROM clean) AS massimo,
    (SELECT AVG(valore_num) FROM clean) AS media,
    (SELECT AVG(valore_num)
     FROM ordered
     WHERE rn IN ((n + 1) / 2, (n + 2) / 2)
    ) AS mediana;
"""

df_punto_5E = pd.read_sql_query(q_punto_5E, conn)
print("-------Profondita_m Statistiche base-------")
print(df_punto_5E)
print()
# -------------------------------------

q_punto_6A = """
SELECT 
LOWER(TRIM(Au_ppm)) AS valore, 
COUNT(*) AS n
FROM campioni
WHERE TRIM(Au_ppm) <> ''
  AND TRIM(Profondita_m) GLOB '*[^0-9.]*'
GROUP BY LOWER(TRIM(Au_ppm))
ORDER BY n DESC, valore;
"""

print("==================================")
print("--------------Punto 6-------------")
print("==================================")
print("\n")
# -------------------------------------

df_punto_6A = pd.read_sql_query(q_punto_6A, conn)
print("-------Valori non numerici-------")
print(df_punto_6A)
print()
# -------------------------------------

q_punto_6B = """
WITH cleaned AS (
    SELECT 
        Au_ppm,
        CAST(REPLACE(TRIM(Au_ppm), ',', '.') AS REAL) AS val_num
    FROM campioni
    WHERE TRIM(Au_ppm) <> '' 
      AND TRIM(Au_ppm) NOT GLOB '*[^0-9.-]*'  -- ✅ ora accetta anche il segno meno
      AND TRIM(Au_ppm) NOT LIKE '%.%.%'
)
SELECT Au_ppm, val_num
FROM cleaned
WHERE val_num < 0 OR val_num > 1000
ORDER BY val_num;
"""

df_punto_6B = pd.read_sql_query(q_punto_6B, conn)
print("-------Outlier Au_ppm-------")
print(df_punto_6B)
print()
# -------------------------------------

q_punto_6C = """
WITH Au_base AS (
    SELECT
        Au_ppm,
        LOWER(TRIM(Au_ppm)) AS v
    FROM campioni
),
Au_parsed AS (
    SELECT
        CASE
            WHEN v IS NULL OR v = '' THEN NULL
            WHEN v GLOB '*[^0-9,.-]*' THEN NULL
            ELSE CAST(REPLACE(v, ',', '.') AS REAL)
        END AS valore_num
    FROM Au_base
),
Au_clean AS (
    SELECT valore_num
    FROM Au_parsed
    WHERE valore_num BETWEEN 0 AND 1000
),
Au_ordered AS (
    SELECT
        valore_num,
        ROW_NUMBER() OVER (ORDER BY valore_num) AS rn,
        COUNT(*) OVER () AS n
    FROM Au_clean
)
SELECT
    (SELECT MIN(valore_num) FROM Au_clean) AS minimo,
    (SELECT MAX(valore_num) FROM Au_clean) AS massimo,
    (SELECT AVG(valore_num) FROM Au_clean) AS media,
    (SELECT AVG(valore_num)
     FROM Au_ordered
     WHERE rn IN ((n + 1) / 2, (n + 2) / 2)
    ) AS mediana;
"""

df_punto_6C = pd.read_sql_query(q_punto_6C, conn)
print("-------Au Statistiche base-------")
print(df_punto_6C)
print()
# -------------------------------------

print("==================================")
print("--------------Punto 7-------------")
print("==================================")
print("\n")
# -------------------------------------

q_punto_7A = """
SELECT
  CASE
    WHEN TRIM(Data) GLOB '[0-9][0-9][0-9][0-9]-[0-9][0-9]-[0-9][0-9]' THEN 'YYYY-MM-DD'
    WHEN TRIM(Data) GLOB '[0-9][0-9]/[0-9][0-9]/[0-9][0-9][0-9][0-9]' THEN 'DD/MM/YYYY'
    WHEN LOWER(TRIM(Data)) = 'oggi' THEN 'oggi'
    ELSE 'altro'
  END AS formato,
  COUNT(*) AS quanti
FROM campioni
GROUP BY formato;
"""

df_punto_7A = pd.read_sql_query(q_punto_7A, conn)
print("-------Analisi Data-------")
print(df_punto_7A)
print()
# -------------------------------------

q_7B = """
SELECT DISTINCT Data
FROM campioni
ORDER BY Sito;
"""

df_7B = pd.read_sql_query(q_7B, conn)
print("-------Valori unici di Data-------")
print(df_7B)
print()
# -------------------------------------

q_7C = """
SELECT 
COUNT(*) AS n,
Data
FROM campioni
WHERE LOWER(TRIM(Data)) LIKE '%oggi%'
ORDER BY Sito;
"""

df_7C = pd.read_sql_query(q_7C, conn)
print("-------Quali record hanno 'oggi'?-------")
print(df_7C)
print()

--------------Punto 1-------------


-------Numero righe-------
   totale_righe
0            50


--------------Punto 2-------------


-------Dettagli Tabella-------
   cid          name  type  notnull dflt_value  pk
0    0   ID_Campione  TEXT        0       None   0
1    1          Sito  TEXT        0       None   0
2    2  Profondita_m  TEXT        0       None   0
3    3        Au_ppm  TEXT        0       None   0
4    4          Data  TEXT        0       None   0


-------Quanti NULL ci sono in Au_ppm / Profondita_m?-------
   totale_null_au_ppm  pct_null_au_ppm  totale_null_profondita_m  \
0                   1              2.0                         0   

   pct_null_profondita_m  
0                    0.0  


--------------Punto 3-------------


-------Pattern di sporcizia su ID_Campione-------
   totale_ID_maiuscolo  totale_ID_minuscolo  totale_spazi  totale_trattino  \
0                   42                    8             3               46   

   totale_NON_trattino  
0   